In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path
from shapely.geometry import box
import earthaccess as ea
from pathlib import Path
import rioxarray
import contextily as ctx
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import Image, display

In [29]:
data_dir = Path("/home/johnny/research/swot_oob_yolo/SWOT_IW_Dataset/Eastern Equatorial Indian/")

nc_files = sorted(data_dir.glob("data/SWOT_L2_LR_SSH_Expert_005_105_20231016T094434_20231016T103603_PGC0_01.nc"))
label_files = sorted(data_dir.glob("labels/SWOT_L2_LR_SSH_Expert_005_105_20231016T094434_20231016T103603_PGC0_01.txt"))

ds_saved = xr.open_mfdataset(
                nc_files,
                combine="nested",
                decode_timedelta=True,
                compat='no_conflicts',
                engine='h5netcdf',
                parallel=True,
            )

ds_saved

<xarray.Dataset> Size: 778kB
Dimensions:  (lat: 940, lon: 69)
Coordinates:
    lat      (lat, lon) float32 259kB dask.array<chunksize=(940, 69), meta=np.ndarray>
    lon      (lat, lon) float32 259kB dask.array<chunksize=(940, 69), meta=np.ndarray>
Data variables:
    ssha     (lat, lon) float32 259kB dask.array<chunksize=(940, 69), meta=np.ndarray>

In [31]:
with open(label_files[0], "r", encoding="utf-8") as f:
    text = f.read()

print(text)

0 0.511157 0.200959 0.915309 0.201136 0.921276 0.127733 0.517123 0.127556
0 0.114195 0.156051 0.607763 0.145937 0.452411 0.105087 -0.041157 0.115201
0 0.620166 0.111309 1.047518 0.098683 0.858409 0.064193 0.431058 0.076819
0 0.518790 0.331532 0.999054 0.331094 0.988476 0.268602 0.508212 0.269040


In [32]:
import numpy as np
import pandas as pd
import plotly.express as px

ds_plot = ds_saved.isel(file=0) if "file" in ds_saved.dims else ds_saved

lat = ds_plot["lat"].values
lon = ds_plot["lon"].values
ssha = ds_plot["ssha"].values

df = pd.DataFrame({
    "lat": lat.ravel(),
    "lon": lon.ravel(),
    "ssha": ssha.ravel()
}).dropna(subset=["lat", "lon", "ssha"])

# Use only the central 95% color range
vmin, vmax = np.nanpercentile(df["ssha"], [2.5, 80])

fig = px.scatter_map(
    df,
    lat="lat",
    lon="lon",
    color="ssha",
    color_continuous_scale="RdBu_r",
    range_color=(vmin, vmax),
    zoom=4,
    height=700,
    opacity=0.8,
    title="Interactive SWOT SSHA Map"
)

fig.update_traces(marker=dict(size=4))

fig.update_layout(
    map_style="open-street-map",
    margin=dict(l=0, r=0, t=40, b=0)
)

fig.show()